In [79]:
import pandas as pd
from glob import glob
import os
from tqdm.notebook import tqdm
from os.path import join, basename, abspath

from instruct_rl.utils.result import get_progress_result_multi, get_progress_result
from instruct_rl.utils.result import process_result


In [80]:
# dataset dir
RESULT_DIR = 'results'

In [81]:
AGG_EVAL_DIRS = {
    'eval_comb2_instruct_pcgrl_0313': {'method': '_cls'},
    'eval_comb2_instruct_pcgrl_03131': {'method': '_st'},
    'eval_comb2_instruct_pcgrl_03132': {'method': '_cls_st'},
    'eval_comb2_instruct_pcgrl_03133': {'method': '_mk'},

}
type(AGG_EVAL_DIRS)

dict

In [1]:
for agg_eval_dir, attrs in AGG_EVAL_DIRS.items():
    eval_path = abspath(join(RESULT_DIR, agg_eval_dir))
    eval_files = glob(join(eval_path, '*', 'raw.csv'))

NameError: name 'AGG_EVAL_DIRS' is not defined

In [83]:
dfs = list()

for agg_eval_dir, attrs in AGG_EVAL_DIRS.items():
    eval_path = abspath(join(RESULT_DIR, agg_eval_dir))
    eval_files = glob(join(eval_path, '*', 'raw.csv'))

    for eval_file in tqdm(eval_files, desc=f'Processing {agg_eval_dir} files'):
        div_file = eval_file.replace('raw.csv', 'diversity.csv')
        
        raw_df = pd.read_csv(eval_file)
        div_df = pd.read_csv(div_file)
    
        df = process_result(raw_df, div_df)
        
        for k, v in attrs.items():
            df[k] = v
        
        dfs.append(df)

# remove where 'config.encoder.model' is sa
df = pd.concat(dfs, ignore_index=True)
# df = df[df['config.encoder.model'] != 'sa']
# df = df[df['seed'] == 1]
df

Processing eval_comb2_instruct_pcgrl_0313 files:   0%|          | 0/3 [00:00<?, ?it/s]

Processing eval_comb2_instruct_pcgrl_03131 files:   0%|          | 0/3 [00:00<?, ?it/s]

Processing eval_comb2_instruct_pcgrl_03132 files:   0%|          | 0/3 [00:00<?, ?it/s]

Processing eval_comb2_instruct_pcgrl_03133 files:   0%|          | 0/3 [00:00<?, ?it/s]

,exp_id,exp_seed,instruct,row_i,reward_enum,cond_region,cond_plength,cond_nblock,cond_nbat,cond_batdir,progress_region,progress_plength,progress_nblock,progress_nbat,progress_batdir,diversity,method
0,embed-bert_inst-scn-2_se-24_enc-mlp_tr-f_model...,2,scn-2_se-24,0,1,25,-1,-1,-1,-1,1.000000e+01,-1.000000,-1.0,-1.0,-1.000000,0.505859,_cls
1,embed-bert_inst-scn-2_se-24_enc-mlp_tr-f_model...,2,scn-2_se-24,1,1,25,-1,-1,-1,-1,8.000000e+00,-1.000000,-1.0,-1.0,-1.000000,0.507812,_cls
2,embed-bert_inst-scn-2_se-24_enc-mlp_tr-f_model...,2,scn-2_se-24,2,1,25,-1,-1,-1,-1,8.000000e+00,-1.000000,-1.0,-1.0,-1.000000,0.508789,_cls
3,embed-bert_inst-scn-2_se-24_enc-mlp_tr-f_model...,2,scn-2_se-24,3,1,25,-1,-1,-1,-1,1.800000e+01,-1.000000,-1.0,-1.0,-1.000000,0.508898,_cls
4,embed-bert_inst-scn-2_se-24_enc-mlp_tr-f_model...,2,scn-2_se-24,4,1,50,-1,-1,-1,-1,3.333333e-08,-1.000000,-1.0,-1.0,-1.000000,0.508898,_cls
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16315,embed-bert_inst-scn-2_se-24_enc-mlp_tr-f_model...,1,scn-2_se-24,1355,25,-1,60,-1,-1,2,-1.000000e+00,47.958066,-1.0,-1.0,16.655844,0.478733,_mk
16316,embed-bert_inst-scn-2_se-24_enc-mlp_tr-f_model...,1,scn-2_se-24,1356,25,-1,60,-1,-1,3,-1.000000e+00,60.087529,-1.0,-1.0,49.833333,0.441732,_mk
16317,embed-bert_inst-scn-2_se-24_enc-mlp_tr-f_model...,1,scn-2_se-24,1357,25,-1,60,-1,-1,3,-1.000000e+00,59.006731,-1.0,-1.0,61.666667,0.429145,_mk
16318,embed-bert_inst-scn-2_se-24_enc-mlp_tr-f_model...,1,scn-2_se-24,1358,25,-1,60,-1,-1,3,-1.000000e+00,65.840298,-1.0,-1.0,39.773810,0.445312,_mk


In [84]:

def aggregate_multi_result(df, first_reward_enum, second_reward_enum, scn_num):
    reward_enum = f'{first_reward_enum}{second_reward_enum}'
    df_1 = get_progress_result(df, f'scn-{scn_num}_se-{reward_enum}', first_reward_enum, return_cond_col=False)
    df_2 = get_progress_result(df, f'scn-{scn_num}_se-{reward_enum}', second_reward_enum, return_cond_col=False)
    # merge df_2 to df_1 with 'method' and 'instruct' columns
    df_merge = df_1.merge(df_2, on=['method', 'instruct'], suffixes=('_1', '_2'))
    return df_merge


In [85]:
t1_e1 = aggregate_multi_result(df, 2, 4, scn_num=1).drop(columns=['task_1', 'task_2'])
t1_e1['eval_instruct'] = 'scn-1_se-24'
t1_e1

,method,instruct,progress_1,progress_2,eval_instruct


In [86]:

t1_e2 = get_progress_result_multi(df, 'scn-1_se-24', 24, return_cond_col=False, return_diversity=False).drop(columns=['progress']).drop(columns=['task_1', 'task_2'])
t1_e2['eval_instruct'] = 'scn-2_se-24'
t1_e2

,method,instruct,progress_1,progress_2,eval_instruct


In [87]:
t2_e1 = aggregate_multi_result(df, 2, 4, scn_num=2).drop(columns=['task_1', 'task_2'])
t2_e1['eval_instruct'] = 'scn-1_se-24'
t2_e1

,method,instruct,progress_1,progress_2,eval_instruct
0,_cls,scn-2_se-24,21.517724,25.206787,scn-1_se-24
1,_st,scn-2_se-24,21.517724,25.206787,scn-1_se-24
2,_cls_st,scn-2_se-24,21.517724,25.206787,scn-1_se-24
3,_mk,scn-2_se-24,21.517724,25.206787,scn-1_se-24


In [88]:
t2_e2 = get_progress_result_multi(df, 'scn-2_se-24', 24, return_cond_col=False, return_diversity=False).drop(columns=['progress']).drop(columns=['task_1', 'task_2'])
t2_e2['eval_instruct'] = 'scn-2_se-24'
t2_e2

,method,instruct,progress_1,progress_2,eval_instruct
0,_cls,scn-2_se-24,23.173145,62.782726,scn-2_se-24
1,_st,scn-2_se-24,23.173145,62.782726,scn-2_se-24
2,_cls_st,scn-2_se-24,23.173145,62.782726,scn-2_se-24
3,_mk,scn-2_se-24,23.173145,62.782726,scn-2_se-24


In [89]:
# stack t1_e1, t1_e2, t2_e1, t2_e2
result_df = pd.concat([t1_e1, t1_e2, t2_e1, t2_e2], ignore_index=True)
# melt with eval_instruct
# rename cont_2c to cont, embed_2c to embed, instruct_2c to instruct, rawobs_2c to rawobs
result_df['method'] = result_df['method'].apply(lambda x: x.replace('_2c', ''))
result_df

,method,instruct,progress_1,progress_2,eval_instruct
0,_cls,scn-2_se-24,21.517724,25.206787,scn-1_se-24
1,_st,scn-2_se-24,21.517724,25.206787,scn-1_se-24
2,_cls_st,scn-2_se-24,21.517724,25.206787,scn-1_se-24
3,_mk,scn-2_se-24,21.517724,25.206787,scn-1_se-24
4,_cls,scn-2_se-24,23.173145,62.782726,scn-2_se-24
5,_st,scn-2_se-24,23.173145,62.782726,scn-2_se-24
6,_cls_st,scn-2_se-24,23.173145,62.782726,scn-2_se-24
7,_mk,scn-2_se-24,23.173145,62.782726,scn-2_se-24


In [90]:
pivot_df = result_df.pivot_table(index=['method', 'instruct'], columns='eval_instruct', values=['progress_1', 'progress_2']).reset_index()
# set multi-level column
pivot_df.columns = ['method', 'instruct', 'scn-1-progress_1', 'scn-2-progress_1', 'scn-1-progress_2', 'scn-2-progress_2']
pivot_df = pivot_df[['method', 'instruct', 'scn-1-progress_1', 'scn-1-progress_2', 'scn-2-progress_1', 'scn-2-progress_2']]
# sort with instruct and method columns

pivot_df['method'] = pd.Categorical(pivot_df['method'], [
                                                        '_cls', '_st', '_cls_st', '_mk'])
pivot_df.sort_values(['instruct', 'method'], inplace=True)
# remove categorical type
pivot_df['method'] = pivot_df['method'].astype(str)
pivot_df 

,method,instruct,scn-1-progress_1,scn-1-progress_2,scn-2-progress_1,scn-2-progress_2
0,_cls,scn-2_se-24,21.517724,25.206787,23.173145,62.782726
3,_st,scn-2_se-24,21.517724,25.206787,23.173145,62.782726
1,_cls_st,scn-2_se-24,21.517724,25.206787,23.173145,62.782726
2,_mk,scn-2_se-24,21.517724,25.206787,23.173145,62.782726


In [91]:
progress_cols = ['scn-1-progress_1', 'scn-1-progress_2', 'scn-2-progress_1', 'scn-2-progress_2']
# 
grouped = pivot_df.groupby(['instruct', 'method']).first()


multi_index_columns = pd.MultiIndex.from_tuples([('scn-1', 'progress_1'), ('scn-1', 'progress_2'),
                                                 ('scn-2', 'progress_1'), ('scn-2', 'progress_2'),
                                                 ], names=['', ''])
grouped.columns = list(multi_index_columns)

grouped

(scn-1, progress_1)  (scn-1, progress_2)  \
instruct    method                                              
scn-2_se-24 _cls               21.517724            25.206787   
            _cls_st            21.517724            25.206787   
            _mk                21.517724            25.206787   
            _st                21.517724            25.206787   

                     (scn-2, progress_1)  (scn-2, progress_2)  
instruct    method                                             
scn-2_se-24 _cls               23.173145            62.782726  
            _cls_st            23.173145            62.782726  
            _mk                23.173145            62.782726  
            _st                23.173145            62.782726

In [92]:
pivot_df.to_csv('rq2_multi_task.csv', index=False)

In [93]:
def move_line(text, from_line, to_line):
    lines = text.split("\n")
    if from_line < 0 or from_line >= len(lines) or to_line < 0 or to_line >= len(lines):
        return text

    line_to_move = lines.pop(from_line)
    lines.insert(to_line, line_to_move)

    return "\n".join(lines)

In [102]:
with open('rq2_multi_task.tex', 'w') as f:
    latex_str = grouped.to_latex(
        caption='Comprehensive result for Encoding Model', position='h',
        float_format="%.2f", multirow=True)

    # header = ' &  & ('scn-1', 'progress_1') & ('scn-1', 'progress_2') & ('scn-2', 'progress_1') & ('scn-2', 'progress_2') \\ instruct & method &  &  &  &  \\'
    latex_str = latex_str.replace('(\'scn-1\', \'progress_1\') & ', r'\multicolumn{2}{c}{$\tau{}_{2+4}$}')
    latex_str = latex_str.replace('(\'scn-1\', \'progress_2\')', '')
    latex_str = latex_str.replace('(\'scn-2\', \'progress_1\') & ', r'\multicolumn{2}{c}{$\tau{}_{2\oplus4}$}')
    latex_str = latex_str.replace('(\'scn-2\', \'progress_2\')', '')
    latex_str = latex_str.replace('(\'scn-2\', \'progress_2\')', '')



    latex_str = latex_str.replace(' &  & \multicolumn{', ' & Test & \multicolumn{')
    latex_str = latex_str.replace('instruct & method &  &  &  &  \\\\',
                                  r'Train &  & $P_{\tau_{2}}$ & $P_{\tau_{4}}$ & $P_{\tau_{2}}$ & $P_{\tau_{4}}$ \\')

    latex_str = latex_str.replace('random', 'Random')    
    latex_str = latex_str.replace('cont', 'Cont.')
    latex_str = latex_str.replace('rawobs', 'Cont. (Global)')
    latex_str = latex_str.replace('embed', 'IPCGRL ($z_{bert}$)')
    latex_str = latex_str.replace('instruct', 'IPCGRL ($z_{enc}$)')

    latex_str = latex_str.replace('_cls_st', 'Class. + St.')
    latex_str = latex_str.replace('_cls', 'Classification')
    latex_str = latex_str.replace('_st', 'St. Reward')
    latex_str = latex_str.replace('_mk', 'Masking')


    latex_str = latex_str.replace('scn-1_se-whole', r'')    
    latex_str = latex_str.replace('scn-1_se-24', r'$\tau{}_{2+4}$')
    latex_str = latex_str.replace('scn-2_se-24', r'$\tau{}_{2\oplus4}$')
    latex_str = latex_str.replace('\\cline{1-6}', '\\midrule')
    latex_str = latex_str.replace('\\midrule\n\\bottomrule', '\\bottomrule')
    latex_str = latex_str.replace('llrrrr', 'p{0.7cm}p{2cm}p{0.8cm}p{0.8cm}p{0.8cm}p{0.8cm}')


    latex_str = move_line(latex_str, 10, 8)
    latex_str = move_line(latex_str, 15, 13)
    
    # insert \centering
    latex_str = latex_str.replace('\\begin{tabular}', '\\centering\n\\begin{tabular}')
    
    print(latex_str)
    
    f.write(latex_str)


\begin{table}[h]
\caption{Comprehensive result for Encoding Model}
\centering
\begin{tabular}{p{0.7cm}p{2cm}p{0.8cm}p{0.8cm}p{0.8cm}p{0.8cm}}
\toprule
 & Test & \multicolumn{2}{c}{$\tau{}_{2+4}$} & \multicolumn{2}{c}{$\tau{}_{2\oplus4}$} \\
Train &  & $P_{\tau_{2}}$ & $P_{\tau_{4}}$ & $P_{\tau_{2}}$ & $P_{\tau_{4}}$ \\
\midrule
\multirow[t]{4}{*}{$\tau{}_{2\oplus4}$} & Classification & 21.52 & 25.21 & 23.17 & 62.78 \\
 & St. Reward & 21.52 & 25.21 & 23.17 & 62.78 \\
 & Class. + St. & 21.52 & 25.21 & 23.17 & 62.78 \\
 & Masking & 21.52 & 25.21 & 23.17 & 62.78 \\
\bottomrule
\end{tabular}
\end{table}

